In [ ]:
!pip install -U datasets huggingface_hub

  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached huggingface_hub-1.30.0-py3-none-any.whl.metadata (16 kB)
  Using cached pyarrow-25.0.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
Using cached huggingface_hub-1.30.0-py3-none-any.whl (796 kB)
Using cached pyarrow-25.0.1-cp313-cp313-manylinux_2_28_x86_64.whl (50.1 MB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from huggingface_hub import notebook_login

notebook_login(skip_if_logged_in=False)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

configs = api.list_repo_tree(
    repo_id="ARTPARK-IISc/Vaani",
    repo_type="dataset",
    recursive=False
)

for item in configs:
    print(item.path)

audio
images
.gitattributes
README.md


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "parquet",
    data_files={
        "train": "hf://datasets/ARTPARK-IISc/Vaani/audio/Hindi/train-00000-of-05620.parquet"
    },
    streaming=True,
)

print(ds)

IterableDatasetDict({
    train: IterableDataset({
        features: ['audio', 'language', 'duration', 'speakerID', 'languagesKnown', 'gender', 'state', 'district', 'pincode', 'stay(years)', 'isTranscriptionAvailable', 'transcript', 'referenceImage', 'speakerImageHash', 'UtteranceSequenceID'],
        num_shards: 1
    })
})


In [ ]:
count = 0

for sample in ds["train"]:
    if sample["isTranscriptionAvailable"] == "Yes" and sample["transcript"].strip():
        print("Language:", sample["language"])
        print("Duration:", sample["duration"])
        print("Transcript:", sample["transcript"])
        print("Speaker:", sample["speakerID"])
        print()

        count += 1

        if count == 5:
            break

print("Found:", count)

Language: Hindi
Duration: 8.472
Transcript: <insect_noise> इस फोटो {photo} मे बहुत सारा पानी भरा हुआ नज़र आ रहा है एक व्यक्ति का सर दिख रहा है उसके साइड {side} मे बहुत सारी नाव खड़ी हुई नज़र आ रही है। उसेक -- </insect_noise>
Speaker: GONO_147761

Language: Hindi
Duration: 7.051
Transcript: <noise> इस चित्र में एक बड़े से जंगल की तस्वीर है जहां पे एक छोटा सा गुहा है सीढ़िया हैं। [breathing] </noise>
Speaker: GONO_147874

Language: Hindi
Duration: 10.046
Transcript: <noise> दीवारों का रंग जो है वो पीला और काला है [breathing] बहुत सारे लाइटो और लैम्पो {lamp} से सजाया गया है नीचे जमीन रंग जो है काला है। </noise>
Speaker: NA

Language: Hindi
Duration: 5.907
Transcript: यहा पर एक बिल्डिंग {building} दिखाई दे रही है बिल्डिंग {building} के सामने कुछ पेड़ लगे हुए हैं जो दिखाई दे रहे हैं।
Speaker: GONO_147961

Language: Hindi
Duration: 4.924
Transcript: यहा पर एक गाड़ी खड़ी हुई है जो दिखाई दे रही है एक गेट {gate} लगा है जो दिख  रहा है।
Speaker: GONO_147961

Found: 5


In [ ]:
total = 0
transcribed = 0
total_duration = 0
transcribed_duration = 0

for sample in ds["train"]:
    total += 1
    total_duration += sample["duration"]

    if (
        sample["isTranscriptionAvailable"] == "Yes"
        and sample["transcript"]
        and sample["transcript"].strip()
    ):
        transcribed += 1
        transcribed_duration += sample["duration"]

print("Total samples:", total)
print("Transcribed samples:", transcribed)

print("Total hours:", total_duration / 3600)
print("Transcribed hours:", transcribed_duration / 3600)

print(
    "Transcribed percentage:",
    100 * transcribed / total
)

Total samples: 1799
Transcribed samples: 100
Total hours: 4.334985555555553
Transcribed hours: 0.23747027777777777
Transcribed percentage: 5.558643690939411


In [ ]:
from datasets import load_dataset
from huggingface_hub import get_token

token = get_token()
print("HF token found:", token is not None)

ds = load_dataset(
    "ARTPARK-IISc/Vaani-transcription-part",
    "Hindi",
    streaming=True,
    token=token,
)

print(ds)

it = iter(ds["train"])
samples = [next(it) for _ in range(10)]

for i, s in enumerate(samples):
    print(f"--- sample {i} ---")
    print("duration:", s.get("duration"))
    print("isTranscriptionAvailable:", s.get("isTranscriptionAvailable"))
    print("transcript:", s.get("transcript"))
    print("speakerID:", s.get("speakerID"))

HF token found: True


Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['audio', 'language', 'gender', 'state', 'district', 'transcript', 'referenceImage'],
        num_shards: 193
    })
    validation: IterableDataset({
        features: ['audio', 'language', 'gender', 'state', 'district', 'transcript', 'referenceImage'],
        num_shards: 29
    })
    test: IterableDataset({
        features: ['audio', 'language', 'gender', 'state', 'district', 'transcript', 'referenceImage'],
        num_shards: 28
    })
})
--- sample 0 ---
duration: None
isTranscriptionAvailable: None
transcript: बहुत ही सुन्दर टमाटर है ।
speakerID: None
--- sample 1 ---
duration: None
isTranscriptionAvailable: None
transcript: यहां पे चारों साइड {side} में --
speakerID: None
--- sample 2 ---
duration: None
isTranscriptionAvailable: None
transcript: सुंदर है <pause> यह हॉल {hall} मे एक --</pause>
speakerID: None
--- sample 3 ---
duration: None
isTranscriptionAvailable: None
transcript: <noise> सामने एक पेड़ दिख र

In [ ]:
import re
from collections import Counter
from datasets import load_dataset
from huggingface_hub import get_token

token = get_token()

ds = load_dataset(
    "ARTPARK-IISc/Vaani-transcription-part",
    "Hindi",
    streaming=True,
    token=token,
)

N = 300  # sample size for train; keep modest since audio decoding over the network takes time

def sample_split(split_name, n):
    it = iter(ds[split_name])
    rows = []
    for _ in range(n):
        try:
            rows.append(next(it))
        except StopIteration:
            break
    return rows

train_sample = sample_split("train", N)

# --- Duration stats (decode audio to get real seconds) ---
durations = []
for r in train_sample:
    arr = r["audio"]["array"]
    sr = r["audio"]["sampling_rate"]
    durations.append(len(arr) / sr)

total_sampled_hours = sum(durations) / 3600
avg_utterance_sec = sum(durations) / len(durations)

print(f"Sampled {len(train_sample)} train examples")
print(f"Avg utterance duration: {avg_utterance_sec:.2f} sec")
print(f"Total duration in this sample: {total_sampled_hours:.4f} hours")

# --- Tag catalog: what annotation patterns actually appear, and how often ---
tag_pattern = re.compile(r"<[^>]+>|\{[^}]*\}|\[[^\]]*\]")
tag_counter = Counter()
transcripts_with_tags = 0

for r in train_sample:
    tags_found = tag_pattern.findall(r["transcript"])
    if tags_found:
        transcripts_with_tags += 1
    for t in tags_found:
        # normalize tag content to its "shape" so {side} and {building} both count as "{...}"
        if t.startswith("<"):
            shape = t.lower().strip("</>").split()[0] if t.strip("</>") else t
            tag_counter[f"<{shape}>"] += 1
        elif t.startswith("{"):
            tag_counter["{english_gloss}"] += 1
        elif t.startswith("["):
            tag_counter[t.lower()] += 1

print(f"\nTranscripts containing at least one tag: {transcripts_with_tags}/{len(train_sample)}")
print("\nTag frequency:")
for tag, count in tag_counter.most_common():
    print(f"  {tag}: {count}")

# --- Basic demographic spread, since we can't check speaker overlap directly ---
print("\nGender distribution (sample):", Counter(r["gender"] for r in train_sample))
print("District spread (sample, top 10):", Counter(r["district"] for r in train_sample).most_common(10))

# --- Light sanity check on validation/test splits (small n, just to see if they look similar) ---
for split in ["validation", "test"]:
    small = sample_split(split, 20)
    dur = [len(r["audio"]["array"]) / r["audio"]["sampling_rate"] for r in small]
    print(f"\n[{split}] sample n={len(small)}, avg duration={sum(dur)/len(dur):.2f}s")
    print(f"[{split}] example transcript: {small[0]['transcript']}")

Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Sampled 300 train examples
Avg utterance duration: 2.71 sec
Total duration in this sample: 0.2255 hours

Transcripts containing at least one tag: 189/300

Tag frequency:
  {english_gloss}: 140
  <noise>: 134
  <pause>: 50
  <static_noise>: 24
  <static>: 22
  <talking>: 10
  [unintelligible]: 9
  [noise]: 9
  <people_talking>: 6
  [inaudible]: 2
  [breathing]: 2
  <child_talking>: 2
  <honking>: 2
  [horn]: 2
  <music>: 2
  [inhaling]: 1
  [whispering]: 1
  [static]: 1

Gender distribution (sample): Counter({'Male': 178, 'Female': 122})
District spread (sample, top 10): [('Patna', 71), ('Palamu', 25), ('Shamli', 22), ('Ranchi', 22), ('Garhwa', 21), ('Lucknow', 18), ('Gaya', 16), ('Kapurthala', 12), ('Deoghar', 10), ('Darbhanga', 9)]

[validation] sample n=20, avg duration=3.05s
[validation] example transcript: <static_noise> यह एक नाइ की दुकान हैं यहाँ पर एक आदमी बाल काट रहे हैं । </static_noise>

[test] sample n=20, avg duration=2.58s
[test] example transcript: <noise> गाड़ी छय एगो कार

In [ ]:
from datasets import load_dataset
from huggingface_hub import get_token
import os

token = get_token()

ds = load_dataset(
    "ARTPARK-IISc/Vaani-transcription-part",
    "Hindi",
    streaming=True,
    token=token,
)

N_TRAIN = 3000   # modest subset for pipeline development, not final training size
N_VAL = 300
N_TEST = 300

# Shuffle with a buffer to reduce the district-order skew we saw in Step 2.
# buffer_size controls how many examples are held in memory to sample from —
# bigger buffer = better shuffling, more RAM/network use up front.
train_stream = ds["train"].shuffle(seed=42, buffer_size=5000)
val_stream = ds["validation"].shuffle(seed=42, buffer_size=1000)
test_stream = ds["test"].shuffle(seed=42, buffer_size=1000)

def materialize(stream, n):
    rows = []
    for i, ex in enumerate(stream):
        if i >= n:
            break
        rows.append({
            "audio_array": ex["audio"]["array"],
            "sampling_rate": ex["audio"]["sampling_rate"],
            "transcript": ex["transcript"],
            "language": ex["language"],
            "gender": ex["gender"],
            "state": ex["state"],
            "district": ex["district"],
        })
    return rows

print("Materializing train subset...")
train_rows = materialize(train_stream, N_TRAIN)
print(f"Got {len(train_rows)} train rows")

print("Materializing validation subset...")
val_rows = materialize(val_stream, N_VAL)
print(f"Got {len(val_rows)} validation rows")

print("Materializing test subset...")
test_rows = materialize(test_stream, N_TEST)
print(f"Got {len(test_rows)} test rows")

# Save to a local Dataset object and to disk (Arrow format) so we don't
# have to re-stream every time we restart the notebook this session.
from datasets import Dataset, DatasetDict

def to_hf_dataset(rows):
    return Dataset.from_dict({
        "audio_array": [r["audio_array"] for r in rows],
        "sampling_rate": [r["sampling_rate"] for r in rows],
        "transcript": [r["transcript"] for r in rows],
        "language": [r["language"] for r in rows],
        "gender": [r["gender"] for r in rows],
        "state": [r["state"] for r in rows],
        "district": [r["district"] for r in rows],
    })

subset = DatasetDict({
    "train": to_hf_dataset(train_rows),
    "validation": to_hf_dataset(val_rows),
    "test": to_hf_dataset(test_rows),
})

os.makedirs("/content/vaani_hindi_subset", exist_ok=True)
subset.save_to_disk("/content/vaani_hindi_subset")

print(subset)
print("\nDistrict spread in materialized train subset:")
from collections import Counter
print(Counter(train_rows[i]["district"] for i in range(len(train_rows))).most_common(10))

Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Materializing train subset...
Got 3000 train rows
Materializing validation subset...
Got 300 validation rows
Materializing test subset...
Got 300 test rows


In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import get_token
import os, gc, time

token = get_token()

ds = load_dataset(
    "ARTPARK-IISc/Vaani-transcription-part",
    "Hindi",
    streaming=True,
    token=token,
)

N_TRAIN, N_VAL, N_TEST = 300, 50, 50   # deliberately small — prove memory stability first

def row_generator(split_name, n, buffer_size):
    stream = ds[split_name].shuffle(seed=42, buffer_size=buffer_size)
    t0 = time.time()
    for i, ex in enumerate(stream):
        if i >= n:
            break
        yield {
            "audio_array": ex["audio"]["array"],
            "sampling_rate": ex["audio"]["sampling_rate"],
            "transcript": ex["transcript"],
            "language": ex["language"],
            "gender": ex["gender"],
            "state": ex["state"],
            "district": ex["district"],
        }
        if (i + 1) % 25 == 0:
            print(f"[{split_name}] {i+1}/{n}, {time.time()-t0:.1f}s")
    print(f"[{split_name}] done in {time.time()-t0:.1f}s")

train_ds = Dataset.from_generator(lambda: row_generator("train", N_TRAIN, 500))
gc.collect()
val_ds = Dataset.from_generator(lambda: row_generator("validation", N_VAL, 100))
gc.collect()
test_ds = Dataset.from_generator(lambda: row_generator("test", N_TEST, 100))
gc.collect()

subset = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})

os.makedirs("/content/vaani_hindi_subset", exist_ok=True)
subset.save_to_disk("/content/vaani_hindi_subset")
print(subset)

from collections import Counter
print(Counter(subset["train"]["district"]).most_common(10))

Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

[train] 25/300, 21.8s
[train] 50/300, 22.2s
[train] 75/300, 22.5s
[train] 100/300, 22.8s
[train] 125/300, 23.1s
[train] 150/300, 23.4s
[train] 175/300, 23.8s
[train] 200/300, 24.3s
[train] 225/300, 24.6s
[train] 250/300, 25.5s
[train] 275/300, 26.8s
[train] 300/300, 27.5s
[train] done in 32.6s


...

[test] 25/50, 13.2s
[test] 50/50, 13.5s
[test] done in 18.5s


Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio_array', 'sampling_rate', 'transcript', 'language', 'gender', 'state', 'district'],
        num_rows: 300
    })
    validation: Dataset({
        features: ['audio_array', 'sampling_rate', 'transcript', 'language', 'gender', 'state', 'district'],
        num_rows: 50
    })
    test: Dataset({
        features: ['audio_array', 'sampling_rate', 'transcript', 'language', 'gender', 'state', 'district'],
        num_rows: 50
    })
})
[('Longding', 35), ('Jamui', 33), ('Kabirdham', 32), ('Katni', 32), ('Budaun', 32), ('Gorakhpur', 29), ('Kapurthala', 28), ('Araria', 28), ('Jahanabad', 22), ('Jamtara', 21)]


In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import get_token
import os, gc, time

token = get_token()

ds = load_dataset(
    "ARTPARK-IISc/Vaani-transcription-part",
    "Hindi",
    streaming=True,
    token=token,
)

N_TRAIN, N_VAL, N_TEST = 800, 100, 100

def row_generator(split_name, n, buffer_size):
    stream = ds[split_name].shuffle(seed=42, buffer_size=buffer_size)
    t0 = time.time()
    for i, ex in enumerate(stream):
        if i >= n:
            break
        yield {
            "audio_array": ex["audio"]["array"],
            "sampling_rate": ex["audio"]["sampling_rate"],
            "transcript": ex["transcript"],
            "language": ex["language"],
            "gender": ex["gender"],
            "state": ex["state"],
            "district": ex["district"],
        }
        if (i + 1) % 100 == 0:
            print(f"[{split_name}] {i+1}/{n}, {time.time()-t0:.1f}s")
    print(f"[{split_name}] done in {time.time()-t0:.1f}s")

train_ds = Dataset.from_generator(lambda: row_generator("train", N_TRAIN, 1000))
gc.collect()
val_ds = Dataset.from_generator(lambda: row_generator("validation", N_VAL, 200))
gc.collect()
test_ds = Dataset.from_generator(lambda: row_generator("test", N_TEST, 200))
gc.collect()

subset = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})

os.makedirs("/content/vaani_hindi_subset", exist_ok=True)
subset.save_to_disk("/content/vaani_hindi_subset")
print(subset)

from collections import Counter
print(Counter(subset["train"]["district"]).most_common(10))

Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

[train] 100/800, 19.8s
[train] 200/800, 21.3s
[train] 300/800, 23.3s
[train] 400/800, 25.7s
[train] 500/800, 27.1s
[train] 600/800, 28.8s
[train] 700/800, 30.3s
[train] 800/800, 31.6s
[train] done in 36.6s


...

[test] 100/100, 14.3s
[test] done in 19.3s


Saving the dataset (0/1 shards):   0%|          | 0/800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio_array', 'sampling_rate', 'transcript', 'language', 'gender', 'state', 'district'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['audio_array', 'sampling_rate', 'transcript', 'language', 'gender', 'state', 'district'],
        num_rows: 100
    })
    test: Dataset({
        features: ['audio_array', 'sampling_rate', 'transcript', 'language', 'gender', 'state', 'district'],
        num_rows: 100
    })
})
[('Longding', 86), ('Kabirdham', 83), ('Budaun', 80), ('Jamui', 80), ('Kapurthala', 79), ('Katni', 78), ('Jahanabad', 75), ('Araria', 72), ('Bilaspur', 47), ('Gorakhpur', 39)]


In [ ]:
from datasets import load_from_disk
import re
from collections import Counter

subset = load_from_disk("/content/vaani_hindi_subset")
train = subset["train"]

# --- 1. Full tag catalog on the larger, shuffled set ---
tag_pattern = re.compile(r"<[^>]+>|\{[^}]*\}|\[[^\]]*\]")
tag_counter = Counter()
raw_tag_counter = Counter()  # exact strings, not normalized, so we see real variants

for t in train["transcript"]:
    for tag in tag_pattern.findall(t):
        raw_tag_counter[tag] += 1

print("Raw tag variants found (exact strings):")
for tag, count in raw_tag_counter.most_common(30):
    print(f"  {tag!r}: {count}")

# --- 2. Show a few full real examples per major tag type, so we can eyeball handling ---
def show_examples(keyword, n=3):
    print(f"\n--- examples containing {keyword!r} ---")
    shown = 0
    for t in train["transcript"]:
        if keyword in t:
            print(" ", repr(t))
            shown += 1
        if shown >= n:
            break

for kw in ["<noise>", "<static", "<pause>", "{", "[unintelligible]", "[inaudible]", "''"]:
    show_examples(kw)

# --- 3. Check for the malformed nesting we saw earlier (</pause></pause>, etc.) ---
malformed = [t for t in train["transcript"] if re.search(r"</\w+>\s*</\w+>", t)]
print(f"\nTranscripts with doubled closing tags: {len(malformed)}")
for t in malformed[:5]:
    print(" ", repr(t))

# --- 4. Empty or near-empty transcripts after tag stripping (sanity check for over-cleaning risk) ---
def strip_all_tags(t):
    t = re.sub(r"<[^>]+>", "", t)      # remove <...> and </...>
    t = re.sub(r"\{[^}]*\}", "", t)    # remove {...}
    t = re.sub(r"\[[^\]]*\]", "", t)   # remove [...]
    return t.strip()

near_empty = [t for t in train["transcript"] if len(strip_all_tags(t)) < 3]
print(f"\nTranscripts that would become near-empty if all tags stripped: {len(near_empty)}")
for t in near_empty[:5]:
    print(" ", repr(t))

Raw tag variants found (exact strings):
  '<noise>': 330
  '</noise>': 330
  '<pause>': 253
  '</pause>': 253
  '[breathing]': 114
  '<static_noise>': 77
  '</static_noise>': 77
  '[inhaling]': 77
  '{color}': 64
  '[unintelligible]': 61
  '{colour}': 58
  '{photo}': 46
  '{light}': 35
  '{white}': 33
  '<static>': 32
  '</static>': 32
  '{बहुत}': 28
  '{side}': 26
  '{building}': 25
  '{gate}': 25
  '{road}': 22
  '<talking>': 22
  '</talking>': 22
  '{red}': 22
  '{black}': 20
  '{पर}': 17
  '{green}': 15
  '{yellow}': 15
  '{car}': 14
  '{blue}': 14

--- examples containing '<noise>' ---
  '<noise> दोनों साइड {side} में लाइटे {light} लगाई गई हैं। </noise>'
  '<noise> है और डिब्बे में क्रोमा बी {croma b} लिखा हुआ है ऊपर में कुछ लाइटें {light} [breathing] भी दिख रही हैं और ऐ_सी {A_C} भी-- </noise>'
  '<noise> पार्क {park} में यहां पर बच्चे खेल रहे हैं। <noise></noise></noise>'

--- examples containing '<static' ---
  '<static_noise> एक बड़ा सा हॉल {hall} दिखाई दे रहा है जिसमें बहुत सार

In [ ]:
from datasets import load_from_disk
import re
import unicodedata

subset = load_from_disk("/content/vaani_hindi_subset")
train = subset["train"]

def clean_transcript(text):
    # Unicode normalization first, so downstream regexes see consistent forms
    text = unicodedata.normalize("NFC", text)

    # Strip {gloss} annotations, keep preceding spoken word untouched
    text = re.sub(r"\{[^}]*\}", " ", text)

    # Strip all <tag> and </tag> markers regardless of name/nesting
    text = re.sub(r"<[^>]+>", " ", text)

    # Strip bracket non-speech-event markers EXCEPT unintelligible/inaudible
    # (those are handled separately as a drop-decision, not stripped in place)
    text = re.sub(r"\[(?!unintelligible|inaudible)[^\]]*\]", " ", text)

    # Collapse whitespace created by removals
    text = re.sub(r"\s+", " ", text).strip()
    return text

def should_drop(original_text):
    return bool(re.search(r"\[unintelligible\]|\[inaudible\]", original_text))

def has_incomplete_marker(original_text):
    return original_text.strip().endswith("--")

# --- Compute impact stats before touching anything ---
total = len(train)
drop_flags = [should_drop(t) for t in train["transcript"]]
incomplete_flags = [has_incomplete_marker(t) for t in train["transcript"]]

n_drop = sum(drop_flags)
n_incomplete = sum(incomplete_flags)
n_both = sum(d and i for d, i in zip(drop_flags, incomplete_flags))

print(f"Total rows: {total}")
print(f"Rows with [unintelligible]/[inaudible] (would drop): {n_drop} ({100*n_drop/total:.1f}%)")
print(f"Rows ending in '--' (incomplete utterance marker): {n_incomplete} ({100*n_incomplete/total:.1f}%)")
print(f"Rows that are both: {n_both}")
print(f"Rows remaining if we drop only unintelligible/inaudible: {total - n_drop} ({100*(total-n_drop)/total:.1f}%)")

# --- Show before/after for a sample of rows, including edge cases ---
print("\n--- Before/after samples ---")
sample_indices = list(range(10))
for i in sample_indices:
    orig = train["transcript"][i]
    cleaned = clean_transcript(orig)
    drop = should_drop(orig)
    print(f"\n[{i}] drop={drop}")
    print(f"  before: {orig!r}")
    print(f"  after:  {cleaned!r}")

# --- Specifically check the mid-word-tag case doesn't merge words wrongly ---
midword_examples = [t for t in train["transcript"] if re.search(r"\S\[unintelligible\]\S", t)]
print(f"\nMid-word-adjacent [unintelligible] cases: {len(midword_examples)}")
for t in midword_examples[:3]:
    print("  raw:  ", repr(t))
    print("  clean:", repr(clean_transcript(t)))

Total rows: 800
Rows with [unintelligible]/[inaudible] (would drop): 52 (6.5%)
Rows ending in '--' (incomplete utterance marker): 77 (9.6%)
Rows that are both: 7
Rows remaining if we drop only unintelligible/inaudible: 748 (93.5%)

--- Before/after samples ---

[0] drop=False
  before: '<child_talking> सामने में एक घर है जो व्हाइट {white} कलर {color} का है। </child_talking>'
  after:  'सामने में एक घर है जो व्हाइट कलर का है।'

[1] drop=False
  before: 'और यहाँ पर मंदिर में लाइटें {lights} लगी हुई है। ऊपर में स्वस्तिक बने हुए हैं।'
  after:  'और यहाँ पर मंदिर में लाइटें लगी हुई है। ऊपर में स्वस्तिक बने हुए हैं।'

[2] drop=False
  before: 'और ई घरवा बहुत दिन का हो गइल बा। ई बहुत रोड़े {road} क किनारे बा का। ई पेंट {paint} -उंट करे के खोजता।'
  after:  'और ई घरवा बहुत दिन का हो गइल बा। ई बहुत रोड़े क किनारे बा का। ई पेंट -उंट करे के खोजता।'

[3] drop=False
  before: '<static_noise> एक बड़ा सा हॉल {hall} दिखाई दे रहा है जिसमें बहुत सारी सीटे {seat} पड़ी हुई है उन सीटों {seat} पे बहुत सारे लोग

In [ ]:
from datasets import load_from_disk
import re
import unicodedata
from collections import Counter

subset = load_from_disk("/content/vaani_hindi_subset")
train = subset["train"]

def clean_transcript(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\{[^}]*\}", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\[(?!unintelligible|inaudible)[^\]]*\]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def should_drop(original_text):
    return bool(re.search(r"\[unintelligible\]|\[inaudible\]", original_text))

# --- Word-count distribution specifically for incomplete ('--') utterances ---
incomplete_rows = [t for t in train["transcript"] if t.strip().endswith("--") and not should_drop(t)]
incomplete_word_counts = [len(clean_transcript(t).split()) for t in incomplete_rows]

print(f"Incomplete-utterance rows (not already dropped): {len(incomplete_rows)}")
print(f"Word count distribution: min={min(incomplete_word_counts)}, "
      f"max={max(incomplete_word_counts)}, "
      f"median={sorted(incomplete_word_counts)[len(incomplete_word_counts)//2]}")
print(Counter(incomplete_word_counts).most_common(15))

print("\nShortest 8 incomplete utterances (candidates for dropping):")
paired = sorted(zip(incomplete_word_counts, incomplete_rows))
for wc, t in paired[:8]:
    print(f"  ({wc} words) {clean_transcript(t)!r}")

print("\nLongest 5 incomplete utterances (candidates for keeping):")
for wc, t in paired[-5:]:
    print(f"  ({wc} words) {clean_transcript(t)!r}")

# --- Now build the actual cleaned dataset with a finalized, explicit rule set ---
MIN_WORDS_IF_INCOMPLETE = 3  # provisional threshold — will adjust based on the distribution above

def process_row(text):
    if should_drop(text):
        return None
    cleaned = clean_transcript(text)
    if text.strip().endswith("--"):
        if len(cleaned.split()) < MIN_WORDS_IF_INCOMPLETE:
            return None
    if len(cleaned) == 0:
        return None
    return cleaned

kept, dropped_unintelligible, dropped_short_incomplete, dropped_empty = 0, 0, 0, 0
cleaned_transcripts = []
keep_mask = []

for t in train["transcript"]:
    if should_drop(t):
        dropped_unintelligible += 1
        keep_mask.append(False)
        continue
    cleaned = clean_transcript(t)
    if t.strip().endswith("--") and len(cleaned.split()) < MIN_WORDS_IF_INCOMPLETE:
        dropped_short_incomplete += 1
        keep_mask.append(False)
        continue
    if len(cleaned) == 0:
        dropped_empty += 1
        keep_mask.append(False)
        continue
    kept += 1
    cleaned_transcripts.append(cleaned)
    keep_mask.append(True)

print(f"\n--- Final pass summary (train split) ---")
print(f"Total: {len(train)}")
print(f"Dropped (unintelligible/inaudible): {dropped_unintelligible}")
print(f"Dropped (too-short incomplete, <{MIN_WORDS_IF_INCOMPLETE} words): {dropped_short_incomplete}")
print(f"Dropped (empty after cleaning): {dropped_empty}")
print(f"Kept: {kept} ({100*kept/len(train):.1f}%)")

Incomplete-utterance rows (not already dropped): 70
Word count distribution: min=5, max=39, median=16
[(10, 6), (8, 6), (25, 5), (13, 4), (22, 4), (11, 4), (5, 3), (18, 3), (14, 3), (23, 3), (12, 3), (31, 3), (30, 2), (19, 2), (15, 2)]

Shortest 8 incomplete utterances (candidates for dropping):
  (5 words) 'अनार संतरा केला सेब --'
  (5 words) 'यहाँ पर जो की --'
  (5 words) 'व्हाइट कलर का पेंट --'
  (7 words) 'पिंक कलर बैगनी कलर व्हाइट कलर --'
  (7 words) 'यहाँ कार धूल रही है जोकि --'
  (8 words) 'अ टेनिस बॉल है। अ स्पोर्ट्स शू --'
  (8 words) 'एक बड़ा सा हॉल है हॉल में --'
  (8 words) 'ऐसे लिखा हुआ हैपेंसिल से यूनिक चीज --'

Longest 5 incomplete utterances (candidates for keeping):
  (31 words) 'सामने एक रोड दिखाई दे रहा है जिसपे ऑटो रिक्शा जाते हुए दिखाई दे रहीं हैं सांत ही बोहोत सारे लोग मोटरसाइकल पे जाते हुए नजर आ रहें हैं। उ --'
  (34 words) 'देख सकते हैं आप यहाँ पर हमें कुछ बच्चे योगा करते दिखाई दे रहे हैं यहाँ पे हमें मिर्रोर दिखाई दे रहा हैजहाँ पे हमे इनका रिफ्लैक्शन दिखाई दे र

In [ ]:
from datasets import load_from_disk, Dataset, DatasetDict
import re
import unicodedata

subset = load_from_disk("/content/vaani_hindi_subset")

def should_drop(original_text):
    return bool(re.search(r"\[unintelligible\]|\[inaudible\]", original_text))

def clean_transcript(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\{[^}]*\}", " ", text)               # strip {gloss}, keep preceding word
    text = re.sub(r"<[^>]+>", " ", text)                  # strip all <tag>/</tag>
    text = re.sub(r"\[(?!unintelligible|inaudible)[^\]]*\]", " ", text)  # strip other bracket tags
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"-{1,2}\s*$", "", text).strip()        # strip trailing -- or - marker
    return text

def clean_split(dataset):
    keep_indices, cleaned_texts = [], []
    n_drop_unintell = 0
    for i, t in enumerate(dataset["transcript"]):
        if should_drop(t):
            n_drop_unintell += 1
            continue
        cleaned = clean_transcript(t)
        if len(cleaned) == 0:
            continue
        keep_indices.append(i)
        cleaned_texts.append(cleaned)
    filtered = dataset.select(keep_indices)
    filtered = filtered.remove_columns(["transcript"]).add_column("transcript", cleaned_texts)
    return filtered, n_drop_unintell

cleaned_splits = {}
for split_name in ["train", "validation", "test"]:
    filtered, n_drop = clean_split(subset[split_name])
    cleaned_splits[split_name] = filtered
    print(f"[{split_name}] {len(subset[split_name])} -> {len(filtered)} "
          f"(dropped {n_drop} unintelligible/inaudible, "
          f"{100*len(filtered)/len(subset[split_name]):.1f}% kept)")

cleaned_dataset = DatasetDict(cleaned_splits)
cleaned_dataset.save_to_disk("/content/vaani_hindi_cleaned")
print(cleaned_dataset)

# Spot-check: confirm no leftover braces/tags/dashes remain anywhere
leftover = [t for t in cleaned_dataset["train"]["transcript"]
            if re.search(r"[{}<>]|\[unintelligible\]|\[inaudible\]|--\s*$", t)]
print(f"\nRows with leftover unwanted markers after cleaning: {len(leftover)}")
for t in leftover[:5]:
    print(" ", repr(t))

Flattening the indices:   0%|          | 0/748 [00:00<?, ? examples/s]

[train] 800 -> 748 (dropped 52 unintelligible/inaudible, 93.5% kept)


Flattening the indices:   0%|          | 0/97 [00:00<?, ? examples/s]

[validation] 100 -> 97 (dropped 3 unintelligible/inaudible, 97.0% kept)


Flattening the indices:   0%|          | 0/97 [00:00<?, ? examples/s]

[test] 100 -> 97 (dropped 3 unintelligible/inaudible, 97.0% kept)


Saving the dataset (0/1 shards):   0%|          | 0/748 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/97 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/97 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio_array', 'sampling_rate', 'language', 'gender', 'state', 'district', 'transcript'],
        num_rows: 748
    })
    validation: Dataset({
        features: ['audio_array', 'sampling_rate', 'language', 'gender', 'state', 'district', 'transcript'],
        num_rows: 97
    })
    test: Dataset({
        features: ['audio_array', 'sampling_rate', 'language', 'gender', 'state', 'district', 'transcript'],
        num_rows: 97
    })
})

Rows with leftover unwanted markers after cleaning: 0


In [ ]:
from datasets import load_from_disk

cleaned = load_from_disk("/content/vaani_hindi_cleaned")

train_districts = set(cleaned["train"]["district"])
val_districts = set(cleaned["validation"]["district"])
test_districts = set(cleaned["test"]["district"])

print(f"Train districts: {len(train_districts)}")
print(f"Validation districts: {len(val_districts)}")
print(f"Test districts: {len(test_districts)}")

print(f"\nDistricts in both train and validation: {len(train_districts & val_districts)}")
print(f"Districts in both train and test: {len(train_districts & test_districts)}")
print(f"Districts in both validation and test: {len(val_districts & test_districts)}")

Train districts: 15
Validation districts: 16
Test districts: 14

Districts in both train and validation: 2
Districts in both train and test: 1
Districts in both validation and test: 8


In [ ]:
!pip install transformers accelerate evaluate jiwer --quiet

import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_from_disk

cleaned = load_from_disk("/content/vaani_hindi_cleaned")

MODEL_ID = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("Device:", device)
print("GPU memory allocated after model load (GB):",
      torch.cuda.memory_allocated() / 1e9 if device == "cuda" else "n/a")

# Take a small real batch and run it through preprocessing
batch = cleaned["train"].select(range(4))

def prepare_example(example):
    audio = example["audio_array"]
    sr = example["sampling_rate"]
    input_features = processor.feature_extractor(
        audio, sampling_rate=sr, return_tensors="pt"
    ).input_features[0]
    labels = processor.tokenizer(example["transcript"]).input_ids
    return {"input_features": input_features, "labels": labels}

prepared = [prepare_example(batch[i]) for i in range(len(batch))]

for i, p in enumerate(prepared):
    print(f"\n[{i}] input_features shape: {p['input_features'].shape}")
    print(f"[{i}] label token ids (first 20): {p['labels'][:20]}")
    print(f"[{i}] decoded back: {processor.tokenizer.decode(p['labels'], skip_special_tokens=True)!r}")
    print(f"[{i}] original transcript: {batch[i]['transcript']!r}")

# Stack into an actual batch (pad labels, features are already fixed-size from the feature extractor)
input_features = torch.stack([p["input_features"] for p in prepared]).to(device)

label_lists = [p["labels"] for p in prepared]
max_len = max(len(l) for l in label_lists)
pad_id = processor.tokenizer.pad_token_id
padded_labels = torch.tensor([
    l + [pad_id] * (max_len - len(l)) for l in label_lists
])
# Mask padding so it's ignored in the loss
labels_for_loss = padded_labels.clone()
labels_for_loss[labels_for_loss == pad_id] = -100
labels_for_loss = labels_for_loss.to(device)

# One real forward pass
model.train()
outputs = model(input_features=input_features, labels=labels_for_loss)
loss = outputs.loss
print(f"\nForward pass successful. Loss: {loss.item():.4f}")
print("GPU memory allocated after forward pass (GB):",
      torch.cuda.memory_allocated() / 1e9 if device == "cuda" else "n/a")

# One backward pass to confirm gradients flow without OOM
loss.backward()
print("Backward pass successful.")
print("GPU memory allocated after backward pass (GB):",
      torch.cuda.memory_allocated() / 1e9 if device == "cuda" else "n/a")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.1 MB/s eta 0:00:00


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Device: cuda
GPU memory allocated after model load (GB): 0.967714816


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



[0] input_features shape: torch.Size([80, 3000])
[0] label token ids (first 20): [50258, 50276, 50359, 50363, 45938, 17937, 48521, 35082, 21981, 48449, 21981, 31945, 8485, 237, 41858, 8485, 246, 25411, 37139, 43372]
[0] decoded back: 'सामने में एक घर है जो व्हाइट कलर का है।'
[0] original transcript: 'सामने में एक घर है जो व्हाइट कलर का है।'

[1] input_features shape: torch.Size([80, 3000])
[1] label token ids (first 20): [50258, 50276, 50359, 50363, 3941, 242, 25411, 8485, 107, 44500, 17937, 3941, 223, 8485, 103, 25411, 48449, 31945, 3941, 99]
[1] decoded back: 'और यहाँ पर मंदिर में लाइटें लगी हुई है। ऊपर में स्वस्तिक बने हुए हैं।'
[1] original transcript: 'और यहाँ पर मंदिर में लाइटें लगी हुई है। ऊपर में स्वस्तिक बने हुए हैं।'

[2] input_features shape: torch.Size([80, 3000])
[2] label token ids (first 20): [50258, 50276, 50359, 50363, 3941, 242, 25411, 8485, 230, 8485, 246, 25411, 3941, 113, 17937, 8485, 105, 44500, 8703, 223]
[2] decoded back: 'और ई घरवा बहुत दिन का हो गइल बा। ई बहु

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/vaani_hindi_data', exist_ok=True)

Mounted at /content/drive


In [ ]:
from datasets import load_dataset, Dataset
from huggingface_hub import get_token
import os, gc, time, glob
import numpy as np

token = get_token()

DRIVE_PATH = "/content/drive/MyDrive/vaani_hindi_data/raw_subset_chunks"
os.makedirs(DRIVE_PATH, exist_ok=True)

def already_done_rows(split_name):
    existing = sorted(glob.glob(f"{DRIVE_PATH}/{split_name}_chunk_*"))
    total = 0
    for path in existing:
        try:
            total += len(Dataset.load_from_disk(path))
        except Exception:
            pass
    return len(existing), total

def materialize_resumable(split_name, total_n, chunk_size, max_chunks_this_run=4):
    n_existing_chunks, n_done = already_done_rows(split_name)
    print(f"[{split_name}] already have {n_done}/{total_n} rows in {n_existing_chunks} chunks")

    if n_done >= total_n:
        print(f"[{split_name}] already complete, skipping")
        return

    ds = load_dataset("ARTPARK-IISc/Vaani-transcription-part", "Hindi", streaming=True, token=token)

    # No shuffle() at all now — sequential read, skip() instead of manual next()-draining
    stream = ds[split_name].skip(n_done)
    stream_iter = iter(stream)

    chunk_idx = n_existing_chunks
    chunks_this_run = 0
    t0 = time.time()

    while n_done < total_n and chunks_this_run < max_chunks_this_run:
        this_chunk_n = min(chunk_size, total_n - n_done)
        rows = []
        for _ in range(this_chunk_n):
            try:
                ex = next(stream_iter)
            except StopIteration:
                break
            rows.append({
                "audio_array": np.asarray(ex["audio"]["array"], dtype=np.float32),
                "sampling_rate": ex["audio"]["sampling_rate"],
                "transcript": ex["transcript"],
                "language": ex["language"],
                "gender": ex["gender"],
                "state": ex["state"],
                "district": ex["district"],
            })

        if not rows:
            break

        chunk_ds = Dataset.from_list(rows)
        chunk_path = f"{DRIVE_PATH}/{split_name}_chunk_{chunk_idx}"
        chunk_ds.save_to_disk(chunk_path)

        n_done += len(rows)
        chunks_this_run += 1
        print(f"[{split_name}] chunk {chunk_idx}: {len(rows)} rows, total {n_done}/{total_n}, {time.time()-t0:.1f}s")

        del rows, chunk_ds
        gc.collect()
        chunk_idx += 1

    print(f"[{split_name}] this run added {chunks_this_run} chunk(s); {n_done}/{total_n} total so far")

materialize_resumable("train", total_n=5000, chunk_size=500, max_chunks_this_run=4)

[train] already have 2000/5000 rows in 4 chunks


Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

...

[train] chunk 5: 500 rows, total 3000/5000, 50.6s


Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

[train] chunk 6: 500 rows, total 3500/5000, 60.5s


Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

[train] chunk 7: 500 rows, total 4000/5000, 70.8s
[train] this run added 4 chunk(s); 4000/5000 total so far


In [ ]:
materialize_resumable("train", total_n=5000, chunk_size=500, max_chunks_this_run=2)

[train] already have 4000/5000 rows in 8 chunks


Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

[train] chunk 8: 500 rows, total 4500/5000, 64.1s


Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

[train] chunk 9: 500 rows, total 5000/5000, 72.4s
[train] this run added 2 chunk(s); 5000/5000 total so far


In [ ]:
materialize_resumable("validation", total_n=500, chunk_size=250, max_chunks_this_run=2)
materialize_resumable("test", total_n=500, chunk_size=250, max_chunks_this_run=2)

[validation] already have 0/500 rows in 0 chunks


Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/250 [00:00<?, ? examples/s]

...

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/250 [00:00<?, ? examples/s]

[test] chunk 0: 250 rows, total 250/500, 16.1s


Saving the dataset (0/1 shards):   0%|          | 0/250 [00:00<?, ? examples/s]

[test] chunk 1: 250 rows, total 500/500, 29.1s
[test] this run added 2 chunk(s); 500/500 total so far
